# Follow-up Adequacy
**We examine censoring rates by treatment year to assess whether patients have sufficient follow-up to produce reliable pseudo-observations at 2 years.**

In [1]:
import numpy as np
import pandas as pd

from flatiron_cleaner import DataProcessorNSCLC

## Import data

In [2]:
dtype_map = pd.read_csv('../outputs/pembrochemo_pembro_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
df = pd.read_csv('../outputs/pembrochemo_pembro_features_df.csv', dtype = dtype_map)

In [3]:
df.shape

(2064, 164)

In [4]:
treatment_df = pd.read_csv('../outputs/pembrochemo_pembro_index.csv')

In [5]:
treatment_df.shape

(20623, 3)

In [6]:
df = pd.merge(df, treatment_df, on = 'PatientID', how = 'left')

In [7]:
df.shape

(2064, 166)

In [8]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [9]:
df['treatment_year'] = df['StartDate'].dt.year

## Start date

In [10]:
df.StartDate.min()

Timestamp('2015-12-25 00:00:00')

## Data lock

In [11]:
# Initialize class 
processor = DataProcessorNSCLC()

mortality_df = processor.process_mortality(file_path = '../data/Enhanced_Mortality_V2.csv',
                                           index_date_df = df, 
                                           index_date_column = 'StartDate',
                                           visit_path = '../data/Visit.csv', 
                                           telemedicine_path = '../data/Telemedicine.csv', 
                                           biomarkers_path = '../data/Enhanced_AdvNSCLCBiomarkers.csv', 
                                           oral_path = '../data/Enhanced_AdvNSCLC_Orals.csv',
                                           progression_path = '../data/Enhanced_AdvNSCLC_Progression.csv',
                                           drop_dates = False)

2026-05-03 15:22:32,837 - INFO - Successfully read Enhanced_Mortality_V2.csv file with shape: (84881, 2) and unique PatientIDs: 84881
2026-05-03 15:22:32,885 - INFO - Successfully merged Enhanced_Mortality_V2.csv df with index_date_df resulting in shape: (2064, 3) and unique PatientIDs: 2064
2026-05-03 15:22:36,888 - INFO - The following columns ['last_visit_date', 'last_biomarker_date', 'last_oral_date', 'last_progression_date'] are used to calculate the last EHR date
2026-05-03 15:22:36,894 - INFO - Successfully processed Enhanced_Mortality_V2.csv file with final shape: (2064, 6) and unique PatientIDs: 2064. There are 0 out of 2064 patients with missing duration values


In [12]:
mortality_df.head(2)

,PatientID,imported_StartDate,DateOfDeath,event,last_ehr_activity,duration
0,FF16F972863F8,2019-01-18,2019-11-15,1,2019-07-22,301.0
1,F5EF114860555,2022-09-30,2023-03-15,1,2023-03-06,166.0


In [13]:
mortality_df['last_date'] = mortality_df[['DateOfDeath', 'last_ehr_activity']].max(axis=1)

In [14]:
data_lock = mortality_df['last_date'].max()

In [15]:
data_lock

Timestamp('2025-11-30 00:00:00')

## Censoring rates by treatment years

In [16]:
df.groupby('treatment_year')['event'].apply(lambda x: (x == 0).mean())

treatment_year
2015    0.000000
2016    0.135593
2017    0.200893
2018    0.275862
2019    0.279167
2020    0.264463
2021    0.343220
2022    0.394850
2023    0.445026
2024    0.599156
2025    0.691919
Name: event, dtype: float64

In [17]:
results = []
for year in sorted(df['treatment_year'].unique()):
    censored = df.query('treatment_year == @year and event == 0', engine='python')
    if len(censored) > 0:
        frac = (censored['duration'] < 730).mean()
        results.append({'year': year, 'frac_censored_before_2y': frac})

pd.DataFrame(results)

,year,frac_censored_before_2y
0,2016,0.250000
1,2017,0.222222
2,2018,0.160714
3,2019,0.149254
4,2020,0.250000
5,2021,0.234568
6,2022,0.260870
7,2023,0.470588
8,2024,1.000000
9,2025,1.000000


**Primary analysis will be restricted to patients treated in 2023 or earlier.**